# **Import Library**


In [51]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

# **Data Loading**


In [52]:
df = pd.read_csv('../assets/online_retail_uci_clustering.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Category,Seasonality,Sales,Cluster,Label
0,538035,84341B,SMALL PINK MAGIC CHRISTMAS TREE,1.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,christmas,Winter,0.85,2,Churn/One Time Spender
1,538035,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
2,538035,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
3,538035,22586,FELTCRAFT HAIRBAND PINK AND BLUE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender
4,538035,22587,FELTCRAFT HAIRBAND RED AND BLUE,2.0,2010-12-09 13:03:00,0.85,16065.0,United Kingdom,others,Winter,1.70,2,Churn/One Time Spender


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318127 entries, 0 to 318126
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    318127 non-null  int64  
 1   StockCode    318127 non-null  object 
 2   Description  318127 non-null  object 
 3   Quantity     318127 non-null  float64
 4   InvoiceDate  318127 non-null  object 
 5   UnitPrice    318127 non-null  float64
 6   CustomerID   318127 non-null  float64
 7   Country      318127 non-null  object 
 8   Category     318127 non-null  object 
 9   Seasonality  318127 non-null  object 
 10  Sales        318127 non-null  float64
 11  Cluster      318127 non-null  int64  
 12  Label        318127 non-null  object 
dtypes: float64(4), int64(2), object(7)
memory usage: 31.6+ MB


# **Feature Engineering**


Content Based Filtering

In [54]:
# Ambil setiap
items = df[['StockCode', 'Description', 'Category']]
items

,StockCode,Description,Category
0,84341B,SMALL PINK MAGIC CHRISTMAS TREE,christmas
1,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,others
2,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,others
3,22586,FELTCRAFT HAIRBAND PINK AND BLUE,others
4,22587,FELTCRAFT HAIRBAND RED AND BLUE,others
...,...,...,...
318122,23145,ZINC T-LIGHT HOLDER STAR LARGE,lighting
318123,22466,FAIRY TALE COTTAGE NIGHT LIGHT,lighting
318124,23275,SET OF 3 HANGING OWLS OLLIE BEAK,decoration
318125,21217,RED RETROSPOT ROUND CAKE TINS,storage


In [55]:
print("Data duplikat:", items.duplicated().sum())
items = items.drop_duplicates().reset_index().drop('index', axis=1)
items.duplicated().sum()

Data duplikat: 314323


np.int64(0)

In [56]:
items['Description'] = items['Description'].apply(lambda desc: desc.strip())
items['Tags'] = items['Description'] + " " + items['Category']
items['Tags'] = items['Tags'].apply(lambda x:x.lower().strip())
items

,StockCode,Description,Category,Tags
0,84341B,SMALL PINK MAGIC CHRISTMAS TREE,christmas,small pink magic christmas tree christmas
1,22566,FELTCRAFT HAIRBAND PINK AND PURPLE,others,feltcraft hairband pink and purple others
2,22565,FELTCRAFT HAIRBANDS PINK AND WHITE,others,feltcraft hairbands pink and white others
3,22586,FELTCRAFT HAIRBAND PINK AND BLUE,others,feltcraft hairband pink and blue others
4,22587,FELTCRAFT HAIRBAND RED AND BLUE,others,feltcraft hairband red and blue others
...,...,...,...,...
3799,90214Z,"LETTER ""Z"" BLING KEY RING",others,"letter ""z"" bling key ring others"
3800,90083,CRYSTAL CZECH CROSS PHONE CHARM,others,crystal czech cross phone charm others
3801,90089,PINK CRYSTAL SKULL PHONE CHARM,others,pink crystal skull phone charm others
3802,72783,BLACK SIL'T SQU CANDLE PLATE,kitchen,black sil't squ candle plate kitchen


# **Vectorizer**


In [57]:
cv = CountVectorizer(stop_words='english')
vector = cv.fit_transform(items['Tags']).toarray()

In [58]:
similarity = cosine_similarity(vector)

# **Inference**


In [59]:
def recommend_item_description(item_name_full, top_n=5):
  index = items[items['Description'] == item_name_full].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

def recommend_item_code(item_code, top_n=5):
  index = items[items['StockCode'] == item_code].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

In [60]:
recommend_item_description('LETTER "Z" BLING KEY RING')

Karena Anda membeli LETTER "Z" BLING KEY RING, mungkin Anda tertarik: 
- LETTER "J" BLING KEY RING (Similarity: 1.0)
- LETTER "D" BLING KEY RING (Similarity: 1.0)
- LETTER "K" BLING KEY RING (Similarity: 1.0)
- LETTER "R" BLING KEY RING (Similarity: 1.0)
- LETTER "G" BLING KEY RING (Similarity: 1.0)


In [61]:
recommend_item_description('RECYCLING BAG RETROSPOT')

Karena Anda membeli RECYCLING BAG RETROSPOT, mungkin Anda tertarik: 
- BOTTLE BAG RETROSPOT (Similarity: 0.75)
- JUMBO BAG RED RETROSPOT (Similarity: 0.6708203932499369)
- RED RETROSPOT SHOPPER BAG (Similarity: 0.6708203932499369)
- RED RETROSPOT CHARLOTTE BAG (Similarity: 0.6708203932499369)
- LUNCH BAG RED RETROSPOT (Similarity: 0.6708203932499369)


In [62]:
items.sample(5)

,StockCode,Description,Category,Tags
1175,22127,PARTY CONES CARNIVAL ASSORTED,others,party cones carnival assorted others
3739,23497,CLASSIC CROME BICYCLE BELL,others,classic crome bicycle bell others
2146,16258A,SWIRLY CIRCULAR RUBBERS IN BAG,bags,swirly circular rubbers in bag bags
356,85152,HAND OVER THE CHOCOLATE SIGN,decoration,hand over the chocolate sign decoration
2806,22104,MIRROR MOSAIC CANDLE PLATE,kitchen,mirror mosaic candle plate kitchen


In [63]:
recommend_item_code('23376')

Karena Anda membeli PACK OF 12 VINTAGE CHRISTMAS TISSUE, mungkin Anda tertarik: 
- PACK OF 12 50'S CHRISTMAS TISSUES (Similarity: 0.7499999999999999)
- PACK OF 12 CHRISTMAS FUN CARDS (Similarity: 0.7499999999999999)
- VINTAGE CHRISTMAS TABLECLOTH (Similarity: 0.7216878364870323)
- VINTAGE CHRISTMAS STOCKING (Similarity: 0.7216878364870323)
- PACK 3 BOXES CHRISTMAS PANNETONE (Similarity: 0.6681531047810608)


In [64]:
recommend_item_code('90129D')

Karena Anda membeli AMBER GLASS TASSLE BAG CHARM, mungkin Anda tertarik: 
- PINK GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- GREEN GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- TURQUOISE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- RED GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- PURPLE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)


In [65]:
recommend_item_code('20756')

Karena Anda membeli GREEN FERN POCKET BOOK, mungkin Anda tertarik: 
- CHRYSANTHEMUM POCKET BOOK (Similarity: 0.5773502691896258)
- GREEN FERN SKETCHBOOK (Similarity: 0.5773502691896258)
- GREEN FERN JOURNAL (Similarity: 0.5773502691896258)
- BLUE PAISLEY POCKET BOOK (Similarity: 0.5)
- ABSTRACT CIRCLES POCKET BOOK (Similarity: 0.5)


In [66]:
def recommend_item_description(item_name_full, top_n=5):
  index = items[items['Description'] == item_name_full].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

def recommend_item_code(item_code, top_n=5):
  index = items[items['StockCode'] == item_code].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

In [67]:
recommend_item_description('LETTER "Z" BLING KEY RING')

Karena Anda membeli LETTER "Z" BLING KEY RING, mungkin Anda tertarik: 
- LETTER "J" BLING KEY RING (Similarity: 1.0)
- LETTER "D" BLING KEY RING (Similarity: 1.0)
- LETTER "K" BLING KEY RING (Similarity: 1.0)
- LETTER "R" BLING KEY RING (Similarity: 1.0)
- LETTER "G" BLING KEY RING (Similarity: 1.0)


In [68]:
recommend_item_description('RECYCLING BAG RETROSPOT')

Karena Anda membeli RECYCLING BAG RETROSPOT, mungkin Anda tertarik: 
- BOTTLE BAG RETROSPOT (Similarity: 0.75)
- JUMBO BAG RED RETROSPOT (Similarity: 0.6708203932499369)
- RED RETROSPOT SHOPPER BAG (Similarity: 0.6708203932499369)
- RED RETROSPOT CHARLOTTE BAG (Similarity: 0.6708203932499369)
- LUNCH BAG RED RETROSPOT (Similarity: 0.6708203932499369)


In [69]:
items.sample(5)

,StockCode,Description,Category,Tags
3209,84551,CROCHET DOG KEYRING,others,crochet dog keyring others
480,21931,JUMBO STORAGE BAG SUKI,bags,jumbo storage bag suki bags
1959,90214D,"LETTER ""D"" BLING KEY RING",others,"letter ""d"" bling key ring others"
3425,90176C,DIAMANTE NECKLACE,others,diamante necklace others
1181,16216,LETTER SHAPE PENCIL SHARPENER,stationery,letter shape pencil sharpener stationery


In [70]:
recommend_item_code('23376')

Karena Anda membeli PACK OF 12 VINTAGE CHRISTMAS TISSUE, mungkin Anda tertarik: 
- PACK OF 12 50'S CHRISTMAS TISSUES (Similarity: 0.7499999999999999)
- PACK OF 12 CHRISTMAS FUN CARDS (Similarity: 0.7499999999999999)
- VINTAGE CHRISTMAS TABLECLOTH (Similarity: 0.7216878364870323)
- VINTAGE CHRISTMAS STOCKING (Similarity: 0.7216878364870323)
- PACK 3 BOXES CHRISTMAS PANNETONE (Similarity: 0.6681531047810608)


In [71]:
recommend_item_code('90129D')

Karena Anda membeli AMBER GLASS TASSLE BAG CHARM, mungkin Anda tertarik: 
- PINK GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- GREEN GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- TURQUOISE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- RED GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- PURPLE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)


In [72]:
recommend_item_code('20756')

Karena Anda membeli GREEN FERN POCKET BOOK, mungkin Anda tertarik: 
- CHRYSANTHEMUM POCKET BOOK (Similarity: 0.5773502691896258)
- GREEN FERN SKETCHBOOK (Similarity: 0.5773502691896258)
- GREEN FERN JOURNAL (Similarity: 0.5773502691896258)
- BLUE PAISLEY POCKET BOOK (Similarity: 0.5)
- ABSTRACT CIRCLES POCKET BOOK (Similarity: 0.5)


In [73]:
sales_by_stockcode = df.groupby(['CustomerID', 'StockCode'])['Sales'].sum()
favorite_items = sales_by_stockcode.groupby('CustomerID').nlargest(5).droplevel(level=1).reset_index()

favorite_items

,CustomerID,StockCode,Sales
0,12747.0,82484,1319.40
1,12747.0,82482,325.20
2,12747.0,82494L,315.60
3,12747.0,85062,208.20
4,12747.0,84879,148.72
...,...,...,...
18530,18287.0,85039B,176.40
18531,18287.0,85039A,139.20
18532,18287.0,72349B,106.32
18533,18287.0,85173,81.12


In [74]:
customer_dict = favorite_items.groupby('CustomerID')['StockCode'].apply(list).to_dict()

customer_dict

{12747.0: ['82484', '82482', '82494L', '85062', '84879'],
 12748.0: ['90154', '90149', '23173', '72803A', '72802C'],
 12749.0: ['22171', '22768', '22720', '22170', '22960'],
 12820.0: ['23313', '23323', '22086', '22592', '84946'],
 12821.0: ['23068', '22295', '22752', '22335', '22156'],
 12822.0: ['22086', '22866', '23439', '22741', '23320'],
 12823.0: ['48138'],
 12824.0: ['85048', '23320', '23083', '23274', '21733'],
 12826.0: ['20712', '21930', '22086', '23200', '20978'],
 12827.0: ['22139', '22138', '84978', '22907', '23096'],
 12828.0: ['22111', '37446', '22865', '22866', '84347'],
 12829.0: ['22797', '84826', '84827', '84616', '22794'],
 12830.0: ['22693', '21891', '21703', '21912', '21790'],
 12831.0: ['22078', '84970L', '84970S', '85066', '22457'],
 12832.0: ['23355', '21755', '23534', '22114', '72807C'],
 12833.0: ['21136', '84879', '22170', '22766', '22763'],
 12834.0: ['21181', '82582', '82600', '21165', '47566'],
 12836.0: ['85123A', '21755', '82484', '85066', '21733'],
 12

In [75]:
def recommend_for_single_customer(idcustomer, top_n=5):
    if idcustomer not in customer_dict:
        print(f"CustomerID {idcustomer} tidak ditemukan")
        
        return
    
    print(f"CustomerID: {idcustomer}")
    
    stock_codes = customer_dict[idcustomer]
    all_similarities = []
    
    for stock_code in stock_codes:
        index = items[items['StockCode'] == stock_code].index[0]
        distances = sorted(list(enumerate(similarity[index])))
        all_similarities.extend(distances)
        
        print(f"- Anda membeli {items.iloc[index]['Description']}", sep="")
        
    
    similarity_dict = {}
    
    for idx, sim in all_similarities:
        if idx in similarity_dict:
            similarity_dict[idx].append(sim)
        else:
            similarity_dict[idx] = [sim]
    
    avg_similarities = []
    
    for idx, sims in similarity_dict.items():
        if items.iloc[idx]['StockCode'] not in stock_codes:
            avg_similarities.append((idx, np.mean(sims)))
    
    avg_similarities = sorted(avg_similarities,reverse=True, key = lambda x: x[1])
    recommendations = []
    
    for i in range(top_n):
        idx, sim_score = avg_similarities[i]
        recommendations.append({
            'StockCode': items.iloc[idx]['StockCode'],
            'Description': items.iloc[idx]['Description'],
            'Similarity': sim_score
        })
    
    print("\nRekomendasi untuk customer ini:")
    
    for i, rec in enumerate(recommendations):
        print(f"{i + 1}. {rec['Description']}")
        print(f"   StockCode: {rec['StockCode']} | Similarity: {rec['Similarity']}")

recommend_for_single_customer(12748)

CustomerID: 12748
- Anda membeli LAZER CUT NECKLACE W PASTEL BEADS
- Anda membeli SILVER FLOWR PINK SHELL NECKLACE
- Anda membeli REGENCY TEAPOT ROSES
- Anda membeli ROSE SCENT CANDLE JEWELLED DRAWER
- Anda membeli VANILLA SCENT CANDLE JEWELLED BOX

Rekomendasi untuk customer ini:
1. ROSE SCENT CANDLE IN JEWELLED BOX
   StockCode: 72802A | Similarity: 0.3333333333333334
2. OCEAN SCENT CANDLE IN JEWELLED BOX
   StockCode: 72802B | Similarity: 0.3000000000000001
3. SET/3 ROSE CANDLE IN JEWELLED BOX
   StockCode: 72807A | Similarity: 0.2666666666666667
4. COFFEE SCENT PILLAR CANDLE
   StockCode: 72122 | Similarity: 0.21908902300206648
5. LAVENDER SCENT CAKE CANDLE
   StockCode: 72225C | Similarity: 0.21908902300206648
